# Validate transl_inv_full position matrix

Run-level regression test for the large SrTiO3 artifacts (not suitable for the ordinary unit-test suite). Compares every complex component of $\langle 0i|r|Rj\rangle$ between a Wannier90 `_r.dat` export and the `.mmn` construction, after putting both in the same convention:

- Wigner-Seitz degeneracies are removed from `seedname_r.dat`;
- orbital-dependent R representatives are selected with `min |R + tau_j - tau_i|`; and
- the cached MMN residual has its R=0 diagonal replaced by the Wannier centres.

The optional reference run is a control: it detects whether adding `transl_inv_full` changed the position matrix written by `wannier90.x`. Wannier90 documents that keyword as a `postw90.x` option, so this control matters when interpreting a failed equivalence check.

Set `RUN_COMPUTATION = False` to load the last saved report from `results/validate_transl_inv_full/report.json` instead of recomputing (the original CLI script only wrote a report when `--report` was passed; this notebook always saves one by default so the reload path has something to load).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import json

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "validation").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from validation._paths import RESULTS
from validation.inputs.validate_transl_inv_full_lib import (
    DEFAULT_OUTPUT,
    DEFAULT_REFERENCE,
    DEFAULT_CACHE,
    PREFIX,
    position_from_cache,
    effective_rdat,
    align_rdat,
    compare_matrices,
    print_comparison,
    run_equivalence_check,
    save_report,
    load_report,
)
from pythtb import W90
from wannier.wannier_io import read_wannier_checkpoint

## Config

Mirrors the old `--output-dir`, `--cache`, `--reference-dir`, `--prefix`, `--max-abs`, `--max-relative`, `--R-distance-tolerance` flags.

In [ ]:
RUN_COMPUTATION = False  # True: rebuild and recompute. False: load results/validate_transl_inv_full/report.json.
OUTPUT_DIR = DEFAULT_OUTPUT       # was --output-dir
CACHE = DEFAULT_CACHE             # was --cache
REFERENCE_DIR = DEFAULT_REFERENCE # was --reference-dir; set to None to skip the control comparison
PREFIX = PREFIX                   # was --prefix
MAX_ABS = 1e-6                    # was --max-abs
MAX_RELATIVE = 1e-6               # was --max-relative
R_DISTANCE_TOLERANCE = 1e-5       # was --R-distance-tolerance
REPORT_PATH = RESULTS / "validate_transl_inv_full" / "report.json"  # was --report (optional there; always used here)

## Load inputs individually

Running these as separate cells (rather than inside one `main()`) keeps `cached_position`, `checkpoint`, `w90`, and the aligned/effective position matrices around as inspectable variables.

In [ ]:
if RUN_COMPUTATION:
    cached_position, cached_centres, cache_metadata = position_from_cache(CACHE)
    checkpoint = read_wannier_checkpoint(OUTPUT_DIR / f"{PREFIX}.chk")
    w90 = W90(str(OUTPUT_DIR), PREFIX)

In [ ]:
if RUN_COMPUTATION:
    output_position = effective_rdat(w90)
    aligned_output = align_rdat(
        output_position,
        np.asarray(checkpoint["real_lattice"]),
        cached_centres,
        tuple(int(v) for v in checkpoint["mp_grid"]),
        R_DISTANCE_TOLERANCE,
    )
    # aligned_output: dict[(R_a, R_b, R_c) -> (3, num_wann, num_wann) complex] -- the position matrix itself

## Compare against the MMN-derived cache

In [ ]:
if RUN_COMPUTATION:
    equivalence = compare_matrices(
        aligned_output, cached_position,
        max_abs_tolerance=MAX_ABS, relative_tolerance=MAX_RELATIVE,
    )
    print_comparison("_r.dat vs MMN-derived position matrix", equivalence)

## Optional control: compare against the no-transl_inv_full reference run

In [ ]:
if RUN_COMPUTATION and REFERENCE_DIR is not None:
    old_w90 = W90(str(Path(REFERENCE_DIR).resolve()), PREFIX)
    reference_comparison = compare_matrices(
        output_position, effective_rdat(old_w90),
        max_abs_tolerance=MAX_ABS, relative_tolerance=MAX_RELATIVE,
    )
    print_comparison("_r.dat vs old no-transl_inv_full control", reference_comparison)
elif RUN_COMPUTATION:
    reference_comparison = None

## Full report (convenience path)

Re-runs the same steps as above through one function so the complete JSON-serializable report (gauge/centre drift, MMN content check, transl_inv_full keyword) can be saved in one call -- redundant with the cells above if you already ran them, but convenient on its own.

In [ ]:
if RUN_COMPUTATION:
    report = run_equivalence_check(
        OUTPUT_DIR, CACHE, REFERENCE_DIR, PREFIX,
        max_abs=MAX_ABS, max_relative=MAX_RELATIVE,
        r_distance_tolerance=R_DISTANCE_TOLERANCE,
    )
    save_report(report, REPORT_PATH)

## Load a previous run (skip if you just computed above)

In [ ]:
if not RUN_COMPUTATION:
    report = load_report(REPORT_PATH)
    print(json.dumps(report, indent=2))

`report["rdat_vs_mmn"]["equivalent"]` is the headline pass/fail; `report["rdat_vs_old_control"]` is `None` when no reference run was compared.

In [ ]:
passed = report["rdat_vs_mmn"]["equivalent"]
print(f"passed = {passed}")